# Phase 1 — Decorators & Closures

**Why this matters for DS/ML:**
- PyTorch uses decorators like `@torch.no_grad()` to disable gradient computation during inference.
- Scikit-learn uses them to validate input arrays.
- You'll use them for timing, logging, and caching in your own ML pipelines.

---

## 1. First-Class Functions & Closures

In Python, functions are first-class objects: you can pass them as arguments, return them from other functions, and store them in variables.

A **closure** is a function that remembers variables from its enclosing scope even after that scope has exited.

In [3]:
# Functions are objects


def greet(name) -> str:
    return f"Hello, {name}!"


say_hello = greet  # assign function to a variable
print(say_hello("John"))  # Hello, John!


def apply(func, value):
    return func(value)  # pass function as argument


print(apply(greet, "World"))  # Hello, World!

Hello, John!
Hello, World!


In [4]:
# Closure: inner function captures outer variable
def make_multiplier(factor):
    def multiply(x):
        return x * factor  # 'factor' is captured from outer scope

    return multiply  # return the inner function


double = make_multiplier(2)
triple = make_multiplier(3)

print(double(5))  # 10
print(triple(5))  # 15


# DS use case: create a custom scaler as a closure
def make_normalizer(min_val, max_val):
    def normalize(x):
        return (x - min_val) / (max_val - min_val)

    return normalize


normalize_age = make_normalizer(min_val=0, max_val=100)
print(normalize_age(25))  # 0.25
print(normalize_age(75))  # 0.75

10
15
0.25
0.75


---
## 2. What is a Decorator?

A decorator is a function that **wraps another function** to extend or modify its behavior without changing the original function's code.

```
@decorator
def my_func(): ...

# is exactly the same as:
my_func = decorator(my_func)
```

In [6]:
# The simplest possible decorator
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print(f">>> Calling {func.__name__}")
        result = func(*args, **kwargs)  # call the original function
        print("<<< Done")
        return result

    return wrapper


@my_decorator
def add(a: int, b: int) -> int:
    return a + b


print(add(3, 4))

>>> Calling add
<<< Done
7


---
## 3. `functools.wraps` — Preserve Metadata

Without `@functools.wraps`, the wrapped function loses its `__name__` and `__doc__`. Always use it when writing decorators.

In [7]:
import functools


def my_decorator_bad(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)

    return wrapper


def my_decorator_good(func):
    @functools.wraps(func)  # preserves __name__, __doc__, __module__
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)

    return wrapper


@my_decorator_bad
def compute_mean(data):
    """Compute the mean of a list."""
    return sum(data) / len(data)


@my_decorator_good
def compute_std(data):
    """Compute the standard deviation of a list."""
    mean = sum(data) / len(data)
    return (sum((x - mean) ** 2 for x in data) / len(data)) ** 0.5


print(
    f"Bad  → name: '{compute_mean.__name__}',  doc: '{compute_mean.__doc__}'"
)  # wrapper, None
print(f"Good → name: '{compute_std.__name__}',   doc: '{compute_std.__doc__}'")

Bad  → name: 'wrapper',  doc: 'None'
Good → name: 'compute_std',   doc: 'Compute the standard deviation of a list.'


---
## 4. Practical Decorators for Data Science

In [8]:
import functools
import time


# --- Timer decorator: profile your DS functions ---
def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[timer] {func.__name__} took {elapsed:.4f}s")
        return result

    return wrapper


@timer
def load_and_process(n):
    """Simulate loading n data points and computing their mean."""
    import numpy as np

    data = np.random.randn(n)
    return data.mean()


mean_1k = load_and_process(1_000)
mean_1m = load_and_process(1_000_000)
print(f"Mean (1k): {mean_1k:.4f}")
print(f"Mean (1M): {mean_1m:.4f}")

[timer] load_and_process took 0.3849s
[timer] load_and_process took 0.0270s
Mean (1k): 0.0377
Mean (1M): 0.0000


In [9]:
import numpy as np


# --- Validator decorator: check inputs before running ML functions ---
def validate_array(func):
    """Ensure the first argument is a non-empty 1-D numpy array."""

    @functools.wraps(func)
    def wrapper(data, *args, **kwargs):
        data = np.asarray(data, dtype=float)
        if data.ndim != 1:
            raise ValueError(f"{func.__name__}: expected 1-D array, got {data.ndim}-D")
        if len(data) == 0:
            raise ValueError(f"{func.__name__}: array must not be empty")
        return func(data, *args, **kwargs)

    return wrapper


@validate_array
def zscore(data):
    """Standardize data to zero mean, unit variance."""
    return (data - data.mean()) / data.std()


scores = [72, 85, 90, 60, 78]
print(zscore(scores))  # works — list is converted to array automatically

try:
    zscore([])  # triggers validation error
except ValueError as e:
    print(f"Caught: {e}")

[-0.47760045  0.76416072  1.24176117 -1.62384153  0.09552009]
Caught: zscore: array must not be empty


In [10]:
# --- Decorator with arguments (factory pattern) ---
# To pass arguments to a decorator, you need an extra layer


def repeat(n_times):
    """Run a function n_times and return the last result."""

    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(n_times):
                result = func(*args, **kwargs)
            return result

        return wrapper

    return decorator


@repeat(n_times=3)
def train_epoch(epoch_num):
    print(f"  Training epoch {epoch_num}...")
    return f"epoch_{epoch_num}_done"


result = train_epoch(1)
print(result)

  Training epoch 1...
  Training epoch 1...
  Training epoch 1...
epoch_1_done


In [11]:
# --- Stacking decorators ---
# Applied bottom-up: @timer applied first, then @validate_array wraps the result


@timer
@validate_array
def compute_percentiles(data, percentiles=(25, 50, 75)):
    """Compute specified percentiles of data."""
    return np.percentile(data, percentiles)


sample = np.random.randn(10_000)
p25, p50, p75 = compute_percentiles(sample)
print(f"Q1={p25:.3f}, Median={p50:.3f}, Q3={p75:.3f}")

[timer] compute_percentiles took 0.0266s
Q1=-0.674, Median=0.015, Q3=0.698


---
## 5. Class-Based Decorators

A class can act as a decorator by implementing `__call__`. Useful when the decorator needs to maintain state.

In [12]:
class CallCounter:
    """Tracks how many times a function is called — useful for debugging training loops."""

    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func = func
        self.call_count = 0

    def __call__(self, *args, **kwargs):
        self.call_count += 1
        return self.func(*args, **kwargs)


@CallCounter
def predict(x):
    """Dummy predict function."""
    return x * 2.5 + 1.0


for val in [1, 2, 3, 4, 5]:
    predict(val)

print(f"predict() was called {predict.call_count} times")

predict() was called 5 times


---
## Summary

| Pattern | Syntax | Use Case |
|---------|--------|----------|
| Basic decorator | `@my_decorator` | Logging, timing, validation |
| With `@functools.wraps` | Inside wrapper | Preserve function metadata |
| Decorator factory | `@repeat(n_times=3)` | Parameterized behavior |
| Stacked decorators | `@timer @validate` | Compose multiple concerns |
| Class-based | `__call__` | Stateful decorators |

**In ML libraries you will see:**
- `@torch.no_grad()` — disable autograd during inference
- `@property` — computed attributes (used in sklearn estimators)
- `@staticmethod`, `@classmethod` — used in model classes